# __RAG__

We will see why RAG exists in the first place. LLMs are compressed, lossy representations of their training data. Foundational models cannot access information beyond the date their training data was collected. This leads to some fundamental failures:
1. __Static Knowledge Cutoffs:__ The model's weights are frozen in time, rendering it incapable of answering queries about recent events.
2. __Model Capacity Limits:__ An LLM acts as a lossy compression algorithm over its training corpus, meaning it cannot memorize high-cardinality, niche, or exact private data.
3. __Lack of Access to Private Data:__ Foundational models are trained on public digital data and do not possess enterprise confidatial or personalized data.

When the LLM is forced to sample from a distribution where the probability mass is highly uncertain (the model does not know the answer), the model generates outputs that are factually incorrect or lack grounding. This is a __hallucination__.

To solve this problem, we inject deterministic, retrieved context into the stochastic generation process. 

### __Definition of RAG__
In autoregressive generation, the model predicts the next token based on the user prompt and the previously generated tokens. RAG alters this fundamental idea by introducing a non-parametric memory variable that represents a set of retrieved documents. The generative distribution is now strictly conditioned on both the prompt and the retrieved context.

The non-parametric variable is obtained through _Maximum Inner Product Search (MIPS)_ or _Cosine Similarity_ in a high-dimensional vector space. Embedding models convert the meaning of words and sentences into multidimensional vectors. Looking at this from a system perspective, RAG introduces an I/O and network bottleneck. We are trading GPU compute time for database latency, impacting the Time-To-First-Token (TTFT).

### __Architecture Flow__
A RAG system requires a very well architected flow of data.
1. __User Query:__ Its the raw string input that arrives at the API.
2. __Query Rewriting:__ The raw query can be very ambiguous. Because of that, we can use an LLM or heuristic to rewrite the query into an optimal search string.
3. __Embedding Model:__ The rewritten query is passed through an encoder to generate a dense vector representation.
4. __Vector/Hybrid Search:__ The query vector is compared against a pre-computed index of document chunks. To ensure scalability, databases partition this data and use Approximate Nearest Neighbor (ANN) algorithms like _HNSW_.
5. __Context Augmentation:__ The top K documents are retrieved. We apply a reranking model (like Cross-Encoder) to sort the documents by relevance.
6. __LLM Inference:__ The context is concatenated with the prompt. The LLM processes this massive context window to generated the final response.
7. __Output Guardrails:__ The generated output is validated to ensure no restricted data is leaked and that the answer is strictly derived from the provided context.

### __Types of RAG__
The goal of a RAG system is to maximize the conditional probability of generating the correct answer given a query and a knowledge base.

The conventional RAG, also called __Naive RAG__, fails becaus it aproximates this probability via _dot product_ (or _cosine similarity_) within a fixed-dimensional latent space. The limit is that mapping complex documents into fixed-size vectors creates information compression, that leads to semantic misalignment. Other limit is related to the model's capacity. The model has a strict limit, where if we inject too much information (context) it will degrade the LLMs __attention__ and hits the limit of GPU memory and network latency.

1. __Naive RAG:__ Is the standard flow. It follows: `Text chunking -> Embedding generation -> Vector Search -> Generation`. It solves the static knowledge cutoff of the LLM training, increases the model capacity limit, and solves the lack of access to private data. But it loses global connections within the document and suffers from a high retrieval rate of irrelevant context due to vector space misalignment for complex questions.
2. __Hybrid RAG:__ It combines _semantic search_ (dense vectors) with _lexical/exact search_ (sparse vectors, like BM25 or TF-IDF). TF-IDF and BM25 saturate term frequency, ensuring that rare words have absolute weights. This mitigates the problem with dense embeddings, which often fail to perform "exact matches" on product ID, jargon, or proper nouns.
3. __Conversational RAG (with memory):__ Incorporates the conversaton state to contextualize the search. Applies the Long-Term Memory pattern. The user query is transformed into history. This is important because the isolated query can produce useless embedding; the LLM needs to rewrite it before going to the Vector DB.
4. __HyDE (Hypothetical Document Embedding):__ Before doing the search, the LLM generates a "fake" hypothetical answer based solely on its internal weights. The system then embeds this fake answer to search for real documents. The reason for that is because "query" and "answer" vectors often lies in distant regions in the space (semantic misalignment). HyDE searches for the "shape" of the answer, not the question.
5. __Adaptative RAG (routing):__ A classifier (smaller LLM or predictive model) acts as a router at the beggining of the flow. It decides if the query requires going to the Vector DB, or if it can answered directly by the LLM's weights, or if it requires Web Search. This cuts latency and computational costs by avoiding making a retrieval when its not necessary.
6. __CRAG (Corrective RAG) and Self-RAG:__ They inject evaluation into the loop, transforming RAG into an iterative system.
    
    __CRAG__ evaluates the relevance of what was retrieved. If its poor, it triggers a parallel Web Search as a fallback or refines the chunks before generation.
    
    __Self-RAG__ uses special reflection tokens. The model evaluates its own generation to understand if the answer is supported by the given context. If detects hallucination, it regenerates the answer.
7. __GraphRAG:__ It abandons blind chunking. Uses LLMs in the data ingestion stage to extract _Entities (Nodes)_ and _Relationships (Edges)_, forming a _Knowledge Graph_. It enables _Multihop Reasoning_, where the system links multiple distinct facts or inference steps across different sources to answer a complex question. If the answer requires connecting a concept from document A to document Z, vector search fails. The graph allows traversing edges. It is frequently combined with hierarchical processing, which progressively summarizes node clusters.
8. __AgenticRAG:__ RAG ceases to be a rigid pipeline and becomes one of several "tools" available to an autonomous agent (Tool Calling and Multiagent Collaboration). The agent receives a goal, creates a reasoning plan (Chain of Thought), decides to call some _tool_, evaluates the return, decides if it needs to call another tool, and composes the final answer.

### __System Design Considerations__
__Consistency and Replication:__ The Vector DB cannot fall out of sync with the source relational database (a data warehouse or data lake). We need Change Data Capture pipelines processes asynchronously (using kafka) to update embeddings of modified documents, rerouting failures and controlling write concurrency.

__Data Drift:__ The data distribution can change. We must implement Data Drift Monitoring to ensure that the embeddings and retrieval strategies are still aligning with the production data.